In [1]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


challenge: /home/madmazoku/project/ml-bootcamp-2026/challenge
device: cuda
dataset: /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset


# Staged Pretrained Training With Adversarial Augmentation

Three small phases: classifier/head on top of a pretrained backbone, then the full model, then the full model with adversarial dataset augmentation. The final comparison uses the same correctly classified sample for every phase and estimates the minimum epsilon that flips each checkpoint.


In [2]:
run_id = "notebook-02-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.02, seed=20)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.05, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.05, seed=102),
}

phase_configs = [
    (
        "head",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=1,
            patience=1,
            batch_size=8,
            learning_rate=(1e-3,),
            resplit_runs=1,
        ),
        False,
        0,
    ),
    (
        "full",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=1,
            patience=1,
            batch_size=8,
            learning_rate=(2e-5,),
            resplit_runs=1,
        ),
        True,
        0,
    ),
    (
        "full_adv",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=2,
            patience=2,
            batch_size=8,
            learning_rate=(1e-5,),
            resplit_runs=1,
            adversarial=True,
            adv_epsilon=0.08,
            adv_steps=2,
        ),
        True,
        0,
    ),
]
phases = tuple(
    ad_safe.PhaseSpec(
        job_index=0,
        phase_index=index,
        prefix=f"mobilenet_{name}",
        name=name,
        requested_seed=10 + index,
        config=config,
        unfreeze_all=unfreeze_all,
        unfreeze_top=unfreeze_top,
        model_filename=f"mobilenet_{name}.pt",
        history_filename=f"mobilenet_{name}_history.png",
        json_filename=f"mobilenet_{name}.json",
    )
    for index, (name, config, unfreeze_all, unfreeze_top) in enumerate(phase_configs)
)
plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=(
        ad_safe.JobSpec(
            job_index=0,
            job_id="mobilenet_v3_small_staged",
            display_name="mobilenet staged",
            backbone="mobilenet_v3_small",
            phases=phases,
        ),
    ),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
final_phase = result.phase_results[-1]
final_model_path = final_phase.model_path
[(phase.prefix, phase.model_path) for phase in result.phase_results]


Using device: cuda
Run id: notebook-02-2026-04-22-12-54-44
Output dir: /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44
Train split: train
Eval splits: val, test
Resume: False
Force: True
Cooldown: {'every_epochs': 0, 'seconds': 0.0, 'gpu_max_temp': 0, 'gpu_resume_temp': 0, 'gpu_temp_check_seconds': 15.0}
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/train...
Using stratified dataset source train: 100/10000 samples (fraction=0.01, seed=20)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 300/10000 samples (fraction=0.03, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 300/10000 samples (fraction=0.03, seed=102)
Setup saved to /hom

Evaluating (val):   0%|          | 0/2 [00:00<?, ?it/s]

Initial val metrics: acc=0.2000 auc=0.3200 nll=0.8286 conf=0.6215 margin=0.2431 wrong_conf=0.6053 safe_recall=0.0000 unsafe_recall=0.4000


Training:   0%|          | 0/11 [00:00<?, ?it/s]

Evaluating (val):   0%|          | 0/2 [00:00<?, ?it/s]

...

Split 1/1 | Epoch 1/1 | global_epoch=1 | train=0.5799 | val=0.4286 | auc=0.5833 | nll=0.9859 | best=0.4286 @ epoch 0 | comparison=better | time=0.39s
Saving model to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv.pt...
Best model already in memory from epoch 1 (score: 0.4286)
Saving model to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv.pt...
Figure saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv_history.png
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv.pt...


Evaluating (val):   0%|          | 0/38 [00:00<?, ?it/s]

Evaluating (test):   0%|          | 0/38 [00:00<?, ?it/s]

JSON saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv.json
Metrics CSV saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/accuracy.csv
Setup saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/setup.json

Results
prefix              val                                                                                             test                                                                                          
------------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
mobilenet_full      acc=0.7600 auc=0.8238 nll=0.5446 avg_wrong_conf=0.7335 safe_recall=0.6733 unsaf

[('mobilenet_head',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_head.pt')),
 ('mobilenet_full',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full.pt')),
 ('mobilenet_full_adv',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full_adv.pt'))]

In [3]:
rows = [
    ad_safe.ModelEvalSpec(path=phase.model_path, row_id=phase.prefix)
    for phase in result.phase_results
]
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(rows),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=8, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=8, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Staged phase comparison",
    )
)


Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 300/10000 samples (fraction=0.03, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 300/10000 samples (fraction=0.03, seed=102)

Evaluating mobilenet_head.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_head.pt...


Evaluating (val):   0%|          | 0/38 [00:00<?, ?it/s]

val: acc=0.7567 auc=0.8220 nll=0.5367 conf=0.7757 margin=0.5515 wrong_conf=0.7164 safe_recall=0.7067 unsafe_recall=0.8067


Evaluating (test):   0%|          | 0/38 [00:00<?, ?it/s]

test: acc=0.7133 auc=0.8029 nll=0.5535 conf=0.7802 margin=0.5604 wrong_conf=0.7078 safe_recall=0.6600 unsafe_recall=0.7667

Evaluating mobilenet_full.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_full.pt...


...

Evaluating (val):   0%|          | 0/38 [00:00<?, ?it/s]

val: acc=0.7600 auc=0.8284 nll=0.5398 conf=0.7887 margin=0.5773 wrong_conf=0.7335 safe_recall=0.6800 unsafe_recall=0.8400


Evaluating (test):   0%|          | 0/38 [00:00<?, ?it/s]

test: acc=0.7067 auc=0.8054 nll=0.5701 conf=0.7851 margin=0.5702 wrong_conf=0.7126 safe_recall=0.6267 unsafe_recall=0.7867

Staged phase comparison
model               val                                                                                             test                                                                                          
------------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
mobilenet_head      acc=0.7567 auc=0.8220 nll=0.5367 avg_wrong_conf=0.7164 safe_recall=0.7067 unsafe_recall=0.8067  acc=0.7133 auc=0.8029 nll=0.5535 avg_wrong_conf=0.7078 safe_recall=0.6600 unsafe_recall=0.7667
mobilenet_full_adv  acc=0.7600 auc=0.8284 nll=0.5398 avg_wrong_conf=0.7335 safe_recall=0.6800 unsafe_recall=0.8400  acc=0.7067 auc=0.8054 nll=0.5701 avg_wrong_conf=0.7126 safe_recall=0.6267 unsafe_recall=0.7867
mobilene

EvaluationRunResult(rows=(MetricsMatrixRow(row_id='mobilenet_head', metrics_by_dataset={'val': ClassificationMetrics(accuracy=0.7566666603088379, auc=0.8219555555555557, nll=0.5366707444190979, avg_conf=0.7757430672645569, avg_margin=0.551486074924469, avg_correct_conf=0.6704351902008057, avg_wrong_conf=0.7163859605789185, safe_recall=0.7066666483879089, unsafe_recall=0.8066666722297668), 'test': ClassificationMetrics(accuracy=0.7133333086967468, auc=0.8029333333333332, nll=0.5535186529159546, avg_conf=0.7801983952522278, avg_margin=0.5603968501091003, avg_correct_conf=0.6610584259033203, avg_wrong_conf=0.707802414894104, safe_recall=0.6600000262260437, unsafe_recall=0.7666666507720947)}, metadata={'model_name': 'mobilenet_head.pt', 'model_path': '/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-02-2026-04-22-12-54-44/mobilenet_head.pt', 'rank': 1}), MetricsMatrixRow(row_id='mobilenet_full_adv', metrics_by_dataset={'val': Classificati

In [ ]:
from IPython.display import display
import torch

test_dataset = ad_safe.load_dataset_source(eval_sources["test"])
class_names = ad_safe.get_dataset_classes(test_dataset)
phase_models = [
    (phase, ad_safe.load_model(phase.model_path).eval())
    for phase in result.phase_results
]

comparison_sample_index = 0
found_all_correct = False
for index in range(min(128, len(test_dataset))):
    image_tensor, label_idx = test_dataset[index]
    true_class_name = class_names[label_idx]
    true_label = ad_safe.CLASS_NAMES.index(true_class_name)
    input_tensor = image_tensor.unsqueeze(0).to(ad_safe.DEVICE)
    predictions = []
    with torch.inference_mode():
        for _, phase_model in phase_models:
            logits = ad_safe.forward_logits(phase_model, input_tensor)
            predictions.append(int(logits.argmax(dim=1).item()))
    if all(prediction == true_label for prediction in predictions):
        comparison_sample_index = index
        found_all_correct = True
        break

print("comparison sample index:", comparison_sample_index)
print("correct for every phase:", found_all_correct)
for _, phase_model in phase_models:
    del phase_model
ad_safe.release_torch_memory()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from torch import nn
import torch

image_tensor, label_idx = test_dataset[comparison_sample_index]
true_class_name = class_names[label_idx]
true_label = ad_safe.CLASS_NAMES.index(true_class_name)
original_tensor = image_tensor.unsqueeze(0).to(ad_safe.DEVICE)
criterion = nn.CrossEntropyLoss()
max_epsilon = 0.16
attack_steps = 4

def attack_minimal(model):
    return ad_safe.generate_adversarial_perturbation(
        model=model,
        x_original=original_tensor,
        criterion=criterion,
        target_labels=torch.tensor([true_label], device=ad_safe.DEVICE),
        strategy=ad_safe.MinimalFlipPgdStrategy(
            max_epsilon=max_epsilon,
            num_steps=attack_steps,
            search_steps=10,
            refinement_steps=6,
        ),
        return_result=True,
    )

fig, axes = plt.subplots(1, 1 + len(result.phase_results), figsize=(4.2 * (1 + len(result.phase_results)), 4.4))
axes[0].imshow(image_tensor.permute(1, 2, 0).numpy())
axes[0].set_title(f"Original\nTrue: {true_class_name}")
axes[0].axis("off")

for axis, phase in zip(axes[1:], result.phase_results):
    phase_model = ad_safe.load_model(phase.model_path)
    phase_model.eval()
    attack = attack_minimal(phase_model)
    epsilon_text = f"{attack.epsilon:.4f}" if attack.attack_success else f">{max_epsilon:.2f}"

    axis.imshow(attack.adversarial_tensor[0].detach().cpu().permute(1, 2, 0).numpy())
    axis.set_title(
        f"{phase.phase_name}\n"
        f"orig: {ad_safe.CLASS_NAMES[attack.original_prediction]} {attack.original_confidence:.2f}\n"
        f"adv: {ad_safe.CLASS_NAMES[attack.prediction]} {attack.confidence:.2f}\n"
        f"min_eps={epsilon_text} Linf={attack.linf:.4f}\n"
        f"RMS={attack.rms:.4f} MAE={attack.mae:.4f}"
    )
    axis.axis("off")
    print(
        phase.phase_name,
        "orig=", ad_safe.CLASS_NAMES[attack.original_prediction], f"{attack.original_confidence:.3f}",
        "adv=", ad_safe.CLASS_NAMES[attack.prediction], f"{attack.confidence:.3f}",
        "true_conf:", f"{attack.original_true_confidence:.3f}->{attack.true_confidence:.3f}",
        "min_eps=", epsilon_text,
        "linf=", f"{attack.linf:.4f}",
        "rms=", f"{attack.rms:.4f}",
        "mae=", f"{attack.mae:.4f}",
    )
    del phase_model
    ad_safe.release_torch_memory()

fig.suptitle(
    f"Same sample adversarial comparison | minimal perturbation | max_epsilon={max_epsilon} steps={attack_steps} | sample={comparison_sample_index}"
)
fig.tight_layout()
ad_safe.save_figure(fig, output_dir / "phase_adversarial_comparison.png")
display(fig)
plt.close(fig)
